<a href="https://colab.research.google.com/github/p0ggerz/multilingual-analysis/blob/main/linguistic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Статистический анализ лингвистической разметки заголовков
**Метрики:** абсолютные частоты, %, IPM (instances per million tokens)  
**Тесты:** корреляция Спирмена (по IPM), U-критерий Манна-Уитни (нормировка по токенам),  
поправка на множественное сравнение – метод Беньямини-Хохберга (FDR)

In [1]:
from google.colab import files
uploaded = files.upload()  # загрузите protokol_razmetki_colab.xlsx

Saving protokol_razmetki_colab.xlsx to protokol_razmetki_colab.xlsx


In [12]:
#загрузка библиотек
import jieba
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, unicodedata, warnings
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.linestyle': ':', 'grid.alpha': 0.5,
}) #параметры для красивого и стандартизированного построения графиков
COLORS = {'RU': 'steelblue', 'ZH': 'crimson'}
N = 100

In [13]:
FILENAME = 'protokol_razmetki_colab.xlsx'

df_raw = pd.read_excel(FILENAME, sheet_name='Протокол разметки', header=2)
df_morph = pd.read_excel(FILENAME, sheet_name='Морфологическая разметка')

# бинарные признаки: '+' - 1, отсутствие плюса - 0
df = df_raw.replace('+', 1)
for col in df.columns[9:]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df_ru = df[df['язык заголовка (RU/ZH) '] == 'RU'].copy()
df_zh = df[df['язык заголовка (RU/ZH) '] == 'ZH'].copy()

print(f'Русский сегмент: {len(df_ru)} заголовков')
print(f'Китайский сегмент: {len(df_zh)} заголовков')
print(f'Всего признаков: {len(df.columns) - 9}')

Русский сегмент: 100 заголовков
Китайский сегмент: 100 заголовков
Всего признаков: 88


## Подсчёт токенов
**Русский язык:** токен = слово.  
**Китайский язык:** токен = иероглиф CJK.

In [14]:
TEXT_COL = 'текст заголовка'

def count_ru_tokens(text):
    if not isinstance(text, str): return 1
    tokens = [w for w in re.split(r'\s+', text.strip()) if w]
    return max(len(tokens), 1)

def count_zh_tokens(text):
  if not isinstance(text, str):
        return 1
  tokens = list(jieba.cut(text, cut_all=False))
  tokens = [t for t in tokens if t.strip()]
  return max(len(tokens), 1)

df_ru['tokens'] = df_ru[TEXT_COL].apply(count_ru_tokens)
df_zh['tokens'] = df_zh[TEXT_COL].apply(count_zh_tokens)

TOTAL_TOKENS_RU = int(df_ru['tokens'].sum())
TOTAL_TOKENS_ZH = int(df_zh['tokens'].sum())

print(f'Суммарно токенов — RU: {TOTAL_TOKENS_RU:,}  |  ZH: {TOTAL_TOKENS_ZH:,}')
print(f'Средняя длина заголовка — RU: {df_ru["tokens"].mean():.1f} сл.'
      f'  |  ZH: {df_zh["tokens"].mean():.1f} иер.')
print()
print('Примеры (RU):')
for _, row in df_ru.head(3).iterrows():
    print(f'  [{row["tokens"]:2d}] {str(row[TEXT_COL])[:65]}')
print('Примеры (ZH):')
for _, row in df_zh.head(3).iterrows():
    print(f'  [{row["tokens"]:2d}] {str(row[TEXT_COL])[:40]}')

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.789 seconds.
DEBUG:jieba:Loading model cost 0.789 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


Суммарно токенов — RU: 885  |  ZH: 1,065
Средняя длина заголовка — RU: 8.8 сл.  |  ZH: 10.7 иер.

Примеры (RU):
  [13] От чего зависят семейные и брачные традиции❓ Рассказываю про лини
  [ 9] Твоя жизнь — твой уникальный стиль. Сотвори его смело.
  [17] настоящее место или трон блудницы в мозге: вот почему женский обр
Примеры (ZH):
  [12] 深圳龙岗万科新开一家超级实惠的零食店啦！
  [11] 深圳美食|酸菜鱼和川菜1997年至今爆火🌶
  [11] 真香❗️1688探店预告反季皮毛一体小香外套


In [ ]:
# глобальный список для BH-коррекции
ALL_MW_RESULTS = []

def add_labels(bars, ax=None):
    ax = ax or plt.gca()
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                    f'{h:.0f}', ha='center', va='bottom', fontsize=8)

def freq_table(df_r, df_c, cols, n=100):
    """Абсолютные частоты, % и IPM."""
    rows = []
    for col in cols:
        abs_r = int(df_r[col].sum())
        abs_c = int(df_c[col].sum())
        rows.append({
            'Признак':    col.replace('\n', ' ').strip(),
            'Абс. RU':    abs_r,
            'Отн. RU, %': round(abs_r / n * 100, 1),
            'IPM RU':     round(abs_r / TOTAL_TOKENS_RU * 1_000_000, 1),
            'Абс. ZH':    abs_c,
            'Отн. ZH, %': round(abs_c / n * 100, 1),
            'IPM ZH':     round(abs_c / TOTAL_TOKENS_ZH * 1_000_000, 1),
        })
    return pd.DataFrame(rows)

def ipm_vectors(df_r, df_c, cols):
    r = np.array([df_r[c].sum() / TOTAL_TOKENS_RU * 1_000_000 for c in cols])
    c = np.array([df_c[c].sum() / TOTAL_TOKENS_ZH * 1_000_000 for c in cols])
    return r, c

def bar_chart(r_vals, zh_vals, labels, title,
              ylabel='IPM (instances per million tokens)', figsize=(13, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(labels))
    w = 0.38
    b_ru = ax.bar(x - w/2, r_vals, w, label='Русский сегмент', color=COLORS['RU'])
    b_zh = ax.bar(x + w/2, zh_vals, w, label='Китайский сегмент', color=COLORS['ZH'])
    add_labels(b_ru, ax); add_labels(b_zh, ax)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace('\n', ' ').strip() for l in labels],
                       rotation=40, ha='right', fontsize=9)
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

def spearman_block(r_ipm, zh_ipm, block_name):
    corr, p = spearmanr(r_ipm, zh_ipm)
    sig = '✓ значима' if p < 0.05 else '✗ незначима'
    print(f'Спирмен [{block_name}]:  r = {corr:.3f},  p = {p:.4f}  → {sig}')
    return corr, p

def mannwhitney_block(df_r, df_c, cols, block_name):
    """U-критерий по нормированным на токены значениям.
    Все результаты накапливаются в ALL_MW_RESULTS для BH-коррекции."""
    print(f'\n─── U-критерий Манна–Уитни: {block_name} ───')
    results = []
    for col in cols:
        r_tok = df_r['tokens'].values.astype(float)
        c_tok = df_c['tokens'].values.astype(float)
        r_data = df_r[col].values.astype(float) / r_tok
        c_data = df_c[col].values.astype(float) / c_tok
        if r_data.sum() == 0 and c_data.sum() == 0:
            continue
        stat, p = mannwhitneyu(r_data, c_data, alternative='two-sided')
        clean = col.replace('\n', ' ').strip()
        entry = {'Блок': block_name, 'Признак': clean,
                 'U': round(stat, 1), 'p-value (raw)': round(p, 6)}
        results.append(entry)
        ALL_MW_RESULTS.append(entry.copy())
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df['Значимость'] = res_df['p-value (raw)'].apply(
            lambda v: '**p < 0.05**' if v < 0.05 else 'p ≥ 0.05')
        display(res_df[['Признак','U','p-value (raw)','Значимость']].style.applymap(
            lambda v: 'font-weight:bold; color:darkred' if '**' in str(v) else '',
            subset=['Значимость']))
    return res_df

def plot_block_scatter(r_ipm, zh_ipm, labels, block_name):
    r_ipm  = np.array(r_ipm,  dtype=float)
    zh_ipm = np.array(zh_ipm, dtype=float)
    if r_ipm.sum() == 0 and zh_ipm.sum() == 0:
        print(f'[{block_name}] — все значения нулевые, график пропущен.')
        return
    corr, p_val = spearmanr(r_ipm, zh_ipm)
    sig = 'значима' if p_val < 0.05 else 'незначима'

    colors = plt.cm.tab20(np.linspace(0, 1, len(labels)))

    fig, (ax, ax_leg) = plt.subplots(
        1, 2,
        figsize=(11, 5),
        gridspec_kw={'width_ratios': [3, 1]}
    )

    scatter_handles = []
    for i, (x, y, lbl) in enumerate(zip(r_ipm, zh_ipm, labels)):
        sc = ax.scatter(x, y, s=90, color=colors[i], zorder=3,
                        edgecolors='white', linewidths=0.6,
                        label=f'{i+1}. {lbl.replace(chr(10), " ").strip()}')
        ax.text(x, y, str(i + 1), fontsize=7, ha='center', va='center',
                color='black', fontweight='bold', zorder=4)
        scatter_handles.append(sc)

    if len(set(r_ipm)) > 1:
        m, b_c = np.polyfit(r_ipm, zh_ipm, 1)
        xl = np.linspace(r_ipm.min(), r_ipm.max(), 200)
        ax.plot(xl, m*xl + b_c, color='red', lw=1.5, linestyle='--',
                label='Линия тренда', zorder=2)

    ax.set_xlabel('Русский сегмент (IPM)', fontsize=10)
    ax.set_ylabel('Китайский сегмент (IPM)', fontsize=10)
    ax.set_title(f'Корреляция (IPM): {block_name}', fontsize=12, pad=12)
    ax.text(0.03, 0.95,
            f'Спирмен r = {corr:.3f}\np = {p_val:.4f}  {sig}',
            transform=ax.transAxes,
            bbox=dict(facecolor='white', edgecolor='lightgray', alpha=0.85),
            fontsize=9, verticalalignment='top')

    ax_leg.axis('off')
    legend_text = '\n'.join(
        f'{i+1}. {lbl.replace(chr(10), " ").strip()}'
        for i, lbl in enumerate(labels)
    )
    ax_leg.text(0.0, 0.98, legend_text,
                transform=ax_leg.transAxes,
                fontsize=8, verticalalignment='top',
                family='monospace',
                bbox=dict(facecolor='#f8f8f8', edgecolor='#cccccc',
                          alpha=0.9, boxstyle='round,pad=0.5'))

    plt.tight_layout()
    plt.show()

---
## Блок 1. Графический / орфографический уровень
### 1.1 Графика и 1.2 Пунктуация

In [ ]:
b1_graphic = [
    'повтор букв',
    'сочетание строчных и прописных букв',
    'капитализация текста',
]
b1_punct = [
    'наличие восклицательных знаков',
    'повтор восклицательных знаков',
    'наличие вопросительных знаков',
    'повтор вопросительных знаков',
    'многоточие',
]
b1_all = b1_graphic + b1_punct

ft1 = freq_table(df_ru, df_zh, b1_all)
print('Блок 1.1–1.2: Графика и пунктуация')
display(ft1)

In [ ]:
r1, c1 = ipm_vectors(df_ru, df_zh, b1_all)
spearman_block(r1, c1, 'Блок 1.1–1.2 Графика + Пунктуация')
mannwhitney_block(df_ru, df_zh, b1_all, 'Блок 1.1–1.2 Графика + Пунктуация');

In [ ]:
bar_chart(r1, c1, b1_all, 'Блок 1.1–1.2: Графика и пунктуация (IPM)')

### 1.3 Эмотиконы

In [ ]:
b1_emoji = [
    'наличие эмодзи',
    'номинативная функция',
    'коммуникативная функция',
    'оценочная функция',
    'эмоционально-экспрессивная функция',
    'реактивная функия',
    'модальная функция',
    'разделительная функция',
]
ft_emoji = freq_table(df_ru, df_zh, b1_emoji)
print('Блок 1.3: Эмотиконы')
display(ft_emoji)

In [ ]:
r_emoji, c_emoji = ipm_vectors(df_ru, df_zh, b1_emoji)
spearman_block(r_emoji, c_emoji, 'Блок 1.3 Эмотиконы')
mannwhitney_block(df_ru, df_zh, b1_emoji, 'Блок 1.3 Эмотиконы');

In [ ]:
bar_chart(r_emoji, c_emoji, b1_emoji, 'Блок 1.3: Эмотиконы и их функции (IPM)')

### 1.4 Хэштеги и 1.5 Смешение кодов

In [ ]:
b1_codes = [
    'наличие хэштегов',
    'математические знаки',
    'использование цифр',
    'форма вида n X-ов',
    'форма вида n Y-x X-ов',
    'форма вида n X-ов, которые…',
    'буквенно-цифровые комбинации',
    'комбинации цифр и иероглифов',
    'буквенно-иероглифические комбинации',
    'буквенно-символьные комбинации',
    'сочетание иероглифов и английских слов',
    'буквенно-иероглифически-цифровые комбинации',
]
ft_codes = freq_table(df_ru, df_zh, b1_codes)
print('Блок 1.4–1.5: Смешение кодов')
display(ft_codes)

In [ ]:
r_codes, c_codes = ipm_vectors(df_ru, df_zh, b1_codes)
spearman_block(r_codes, c_codes, 'Блок 1.4–1.5 Смешение кодов')
mannwhitney_block(df_ru, df_zh, b1_codes, 'Блок 1.4–1.5 Смешение кодов');

In [ ]:
bar_chart(r_codes, c_codes, b1_codes, 'Блок 1.4–1.5: Смешение кодов (IPM)')

---
## Блок 2. Фонетический уровень

In [ ]:
b2 = [
    'звукоподражания иероглифами',
    'звукоподражания цифрами',
    'цифровые слова-созвучия',
    'созвучия на основе китайских словосочетаний',
]
ft2 = freq_table(df_ru, df_zh, b2)
print('Блок 2: Фонетический уровень')
display(ft2)

In [ ]:
r2, c2 = ipm_vectors(df_ru, df_zh, b2)
spearman_block(r2, c2, 'Блок 2 Фонетический')
mannwhitney_block(df_ru, df_zh, b2, 'Блок 2 Фонетический');

In [ ]:
bar_chart(r2, c2, b2, 'Блок 2: Фонетический уровень (IPM)')

---
## Блок 3. Морфологический уровень

In [ ]:
df_morph_ru = df_morph[df_morph['language'] == 'русский']
df_morph_zh = df_morph[df_morph['language'] == 'китайский']

ALL_POS = ['NOUN','VERB','ADJ','PROPN','ADV','NUM','PART','ADP','CCONJ','SCONJ','DET','PRON','X']
pos_ru = df_morph_ru['top_pos_en'].value_counts().reindex(ALL_POS, fill_value=0)
pos_zh = df_morph_zh['top_pos_en'].value_counts().reindex(ALL_POS, fill_value=0)
mask = (pos_ru + pos_zh) > 0
pos_ru_f = pos_ru[mask]; pos_zh_f = pos_zh[mask]

POS_RU_NAMES = {'NOUN':'Сущ.','VERB':'Глагол','ADJ':'Прил.','PROPN':'Имя собств.',
                'ADV':'Наречие','NUM':'Числит.','PART':'Частица','X':'X/Прочее'}
pos_labels = [POS_RU_NAMES.get(p, p) for p in pos_ru_f.index]

pos_main_ru = df_ru['преобладающая часть речи'].value_counts()
pos_main_zh = df_zh['преобладающая часть речи'].value_counts()

pos_ipm_ru = pos_ru_f.values / TOTAL_TOKENS_RU * 1_000_000
pos_ipm_zh = pos_zh_f.values / TOTAL_TOKENS_ZH * 1_000_000

pos_df = pd.DataFrame({
    'POS': pos_ru_f.index,
    'Назв.': pos_labels,
    'Абс. RU': pos_ru_f.values,
    'IPM RU': pos_ipm_ru.round(1),
    'Абс. ZH': pos_zh_f.values,
    'IPM ZH': pos_ipm_zh.round(1),
})
print('=== Преобладающая часть речи ===')
display(pos_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(pos_labels))
w = 0.36
b_ru_b = ax.bar(x - w/2, pos_ipm_ru, w, label='Русский сегмент', color=COLORS['RU'])
b_zh_b = ax.bar(x + w/2, pos_ipm_zh, w, label='Китайский сегмент', color=COLORS['ZH'])
add_labels(b_ru_b, ax); add_labels(b_zh_b, ax)
ax.set_title('Преобладающие части речи (IPM)', pad=20, fontsize=14)
ax.set_xlabel('Часть речи'); ax.set_ylabel('IPM')
ax.set_xticks(x); ax.set_xticklabels(pos_labels, fontsize=10)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
b3_morph = [
    'модель: имя прилагательное + имя существительное',
    '1 л. ед.ч.',
    '1 л. мн.ч.',
    'императив',
    'превосходная степень сравнения',
]
for c in b3_morph:
    df_ru[c] = pd.to_numeric(df_ru[c].replace({None: 0}), errors='coerce').fillna(0)
    df_zh[c] = pd.to_numeric(df_zh[c].replace({None: 0}), errors='coerce').fillna(0)

ft3 = freq_table(df_ru, df_zh, b3_morph)
print('Блок 3: Морфологические модели')
display(ft3)

In [ ]:
r3, c3 = ipm_vectors(df_ru, df_zh, b3_morph)
spearman_block(r3, c3, 'Блок 3 Морфология (формы)')
mannwhitney_block(df_ru, df_zh, b3_morph, 'Блок 3 Морфология (формы)');

In [ ]:
bar_chart(r3, c3, b3_morph, 'Блок 3: Морфологические модели (IPM)')

---
## Блок 4. Лексико-семантический уровень

In [ ]:
b4_style = [
    'нейтральная лексика',
    'разговорная лексика',
    'интернет-сленг',
    'сленг',
    'жаргонизм',
    'междометие',
]
ft4s = freq_table(df_ru, df_zh, b4_style)
print('4.1 Стилистически маркированная лексика')
display(ft4s)

In [ ]:
r4s, c4s = ipm_vectors(df_ru, df_zh, b4_style)
spearman_block(r4s, c4s, 'Блок 4.1 Стилистически маркированная лексика')
mannwhitney_block(df_ru, df_zh, b4_style, 'Блок 4.1 Стилистически маркированная лексика');

In [ ]:
bar_chart(r4s, c4s, b4_style, 'Блок 4.1: Стилистически маркированная лексика (IPM)')

In [ ]:
b4_borr = [
    'англицизм',
    'фонетические',
    'семантические',
    'фонетико-семантические',
    'неассимилировное заимствование',
]
ft4b = freq_table(df_ru, df_zh, b4_borr)
print('4.2 Заимствования')
display(ft4b)

In [ ]:
r4b, c4b = ipm_vectors(df_ru, df_zh, b4_borr)
spearman_block(r4b, c4b, 'Блок 4.2 Заимствования')
mannwhitney_block(df_ru, df_zh, b4_borr, 'Блок 4.2 Заимствования');

In [ ]:
bar_chart(r4b, c4b, b4_borr, 'Блок 4.2: Заимствования (IPM)')

In [ ]:
b4_eval = [
    'оценочное прилагательное',
    'интенсификаторы',
    'метафора',
    'гипербола',
    'оксюморон',
    'фразеологизм / идиома',
    'кавычки',
]
ft4e = freq_table(df_ru, df_zh, b4_eval)
print('4.3 Оценочность')
display(ft4e)

In [ ]:
r4e, c4e = ipm_vectors(df_ru, df_zh, b4_eval)
spearman_block(r4e, c4e, 'Блок 4.3 Оценочность')
mannwhitney_block(df_ru, df_zh, b4_eval, 'Блок 4.3 Оценочность');

In [ ]:
bar_chart(r4e, c4e, b4_eval, 'Блок 4.3: Средства оценочности (IPM)')

In [ ]:
b4_add = ['неологизм ', 'аббревиатура']
ft4a = freq_table(df_ru, df_zh, b4_add)
print('4.4 Дополнительная лексика')
display(ft4a)
mannwhitney_block(df_ru, df_zh, b4_add, 'Блок 4.4 Доп. лексика');

---
## Блок 5. Синтаксический уровень

In [ ]:
b5_struct = ['простое','ССП','СПП','БСП','номинативное','неполное']
ft5s = freq_table(df_ru, df_zh, b5_struct)
print('5.1 Тип предложения по структуре')
display(ft5s)

In [ ]:
r5s, c5s = ipm_vectors(df_ru, df_zh, b5_struct)
spearman_block(r5s, c5s, 'Блок 5.1 Тип по структуре')
mannwhitney_block(df_ru, df_zh, b5_struct, 'Блок 5.1 Тип предложения по структуре');

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(b5_struct)); w = 0.36
b_ru_b = ax.bar(x - w/2, r5s, w, label='Русский сегмент', color=COLORS['RU'])
b_zh_b = ax.bar(x + w/2, c5s, w, label='Китайский сегмент', color=COLORS['ZH'])
add_labels(b_ru_b, ax); add_labels(b_zh_b, ax)
ax.set_title('Блок 5.1: Типы предложений по структуре (IPM)', pad=20, fontsize=14)
ax.set_ylabel('IPM'); ax.set_xticks(x)
ax.set_xticklabels([c.replace('\n','').strip().upper() for c in b5_struct], fontsize=10)
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

In [ ]:
b5_goal = ['повествовательное','вопросительное','побудительное']
ft5g = freq_table(df_ru, df_zh, b5_goal)
print('5.2 Тип предложения по цели')
display(ft5g)

In [ ]:
r5g, c5g = ipm_vectors(df_ru, df_zh, b5_goal)
spearman_block(r5g, c5g, 'Блок 5.2 Тип по цели высказывания')
mannwhitney_block(df_ru, df_zh, b5_goal, 'Блок 5.2 Тип предложения по цели');

In [ ]:
goal_labels = ['Повествовательное','Вопросительное','Побудительное']
fig, ax = plt.subplots(figsize=(9, 6)); x = np.arange(3); w = 0.36
b_ru_b = ax.bar(x - w/2, r5g, w, label='Русский сегмент', color=COLORS['RU'])
b_zh_b = ax.bar(x + w/2, c5g, w, label='Китайский сегмент', color=COLORS['ZH'])
add_labels(b_ru_b, ax); add_labels(b_zh_b, ax)
ax.set_title('Блок 5.2: Типы предложений по цели высказывания (IPM)', pad=20, fontsize=14)
ax.set_ylabel('IPM'); ax.set_xticks(x)
ax.set_xticklabels(goal_labels, fontsize=11); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
b5_expr = [
    'парцелляция','сегментация','асиндетон',
    'синтаксический параллелизм','риторический вопрос',
    '«двойной» заголовок','одинаковый синтаксический рисунок',
    'однородные члены',
]
ft5e = freq_table(df_ru, df_zh, b5_expr)
print('Конструкции экспрессивного синтаксиса')
display(ft5e)

In [ ]:
r5e, c5e = ipm_vectors(df_ru, df_zh, b5_expr)
spearman_block(r5e, c5e, 'Блок 5.3 Конструкции экспрессивного синтаксиса')
mannwhitney_block(df_ru, df_zh, b5_expr, 'Блок 5.3 Конструкции экспрессивного синтаксиса');

In [ ]:
expr_labels = ['Парцелляция','Сегментация','Асиндетон','Синт. параллелизм',
               'Рит. вопрос','«Двойной» заголовок','Один. синт. рисунок','Однородные члены']
fig, ax = plt.subplots(figsize=(13, 6)); x = np.arange(len(expr_labels)); w = 0.36
b_ru_b = ax.bar(x - w/2, r5e, w, label='Русский сегмент', color=COLORS['RU'])
b_zh_b = ax.bar(x + w/2, c5e, w, label='Китайский сегмент', color=COLORS['ZH'])
add_labels(b_ru_b, ax); add_labels(b_zh_b, ax)
ax.set_title('Блок 5.3: Конструкции экспрессивного синтаксиса (IPM)', pad=20, fontsize=14)
ax.set_ylabel('IPM'); ax.set_xticks(x)
ax.set_xticklabels(expr_labels, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

---
## Блок 6. Прагматический уровень

In [ ]:
b6_strat = [
    'привлечение внимания','провокация','самопрезентация',
    'адресация читателя','информирование','оценочность',
    'побуждение к действию',
]
ft6s = freq_table(df_ru, df_zh, b6_strat)
print('6.1 Коммуникативная стратегия')
display(ft6s)

In [ ]:
r6s, c6s = ipm_vectors(df_ru, df_zh, b6_strat)
spearman_block(r6s, c6s, 'Блок 6.1 Коммуникативная стратегия')
mannwhitney_block(df_ru, df_zh, b6_strat, 'Блок 6.1 Коммуникативная стратегия');

In [ ]:
strat_labels = ['Привлечение внимания','Провокация','Самопрезентация',
                'Адресация читателя','Информирование','Оценочность',
                'Побуждение к действию']
bar_chart(r6s, c6s, strat_labels, 'Блок 6.1: Коммуникативные стратегии (IPM)')

In [ ]:
b6_pers  = ['личное обращение','1 л. автора']
b6_inter = [
    'прецедентная ситуация','прецедентный текст',
    'прецедентное имя','прецедентное высказывание',
]
b6_all = b6_pers + b6_inter
ft6pi = freq_table(df_ru, df_zh, b6_all)
print('6.2–6.3 Персонализация и интертекстуальность')
display(ft6pi)

In [ ]:
r6pi, c6pi = ipm_vectors(df_ru, df_zh, b6_all)
spearman_block(r6pi, c6pi, 'Блок 6.2–6.3 Персонализация + Интертекстуальность')
mannwhitney_block(df_ru, df_zh, b6_all, 'Блок 6.2–6.3 Персонализация + Интертекстуальность');

In [ ]:
bar_chart(r6pi, c6pi, b6_all, 'Блок 6.2–6.3: Персонализация и интертекстуальность (IPM)')

---
## Сводная таблица всех признаков (IPM)

In [ ]:
all_features = (
    b1_graphic + b1_punct + b1_emoji + b1_codes +
    b2 + b3_morph + b4_style + b4_borr + b4_eval + b4_add +
    b5_struct + b5_goal + b5_expr + b6_strat + b6_pers + b6_inter
)

print(f'Всего признаков в анализе: {len(all_features)}')
ft_all = freq_table(df_ru, df_zh, all_features)
display(ft_all.style
        .bar(subset=['IPM RU'], color='#4682B4', vmin=0)
        .bar(subset=['IPM ZH'], color='#DC143C', vmin=0)
        .format({'IPM RU':'{:.1f}','IPM ZH':'{:.1f}',
                 'Отн. RU, %':'{:.1f}','Отн. ZH, %':'{:.1f}'}))

---
## Корреляция Спирмена: что показывает

Корреляция Спирмена вычисляется по **IPM-векторам блока** — то есть сравниваются не отдельные заголовки, а профили частот признаков двух корпусов с поправкой на длину текстов. Высокая положительная корреляция означает, что иерархия признаков внутри блока **схожа** в русском и китайском сегментах (одни явления частотны в обоих, другие редки в обоих), но ничего не говорит о том, равны ли абсолютные частоты. Различия в частотах выявляет U-критерий Манна–Уитни.

Шкала интерпретации (Чеддок): |r| < 0.3 — слабая; 0.3–0.5 — умеренная; 0.5–0.7 — заметная; 0.7–0.9 — высокая; > 0.9 — весьма высокая.


---
## Корреляция Спирмена по блокам (сводка)

In [ ]:
block_data = {
    '1.1–1.2 Графика+Пункт.':   ipm_vectors(df_ru, df_zh, b1_all),
    '1.3 Эмотиконы':             ipm_vectors(df_ru, df_zh, b1_emoji),
    '1.4–1.5 Смешение кодов':    ipm_vectors(df_ru, df_zh, b1_codes),
    '2. Фонетика':                ipm_vectors(df_ru, df_zh, b2),
    '3. Морфология (формы)':      ipm_vectors(df_ru, df_zh, b3_morph),
    '4.1 Стиль. лексика':         ipm_vectors(df_ru, df_zh, b4_style),
    '4.2 Заимствования':          ipm_vectors(df_ru, df_zh, b4_borr),
    '4.3 Оценочность':            ipm_vectors(df_ru, df_zh, b4_eval),
    '4.4 Доп. лексика':           ipm_vectors(df_ru, df_zh, b4_add),
    '5.1 Тип по структуре':       ipm_vectors(df_ru, df_zh, b5_struct),
    '5.2 Тип по цели':            ipm_vectors(df_ru, df_zh, b5_goal),
    '5.3 Экспресс. синтаксис':    ipm_vectors(df_ru, df_zh, b5_expr),
    '6.1 Коммун. стратегия':      ipm_vectors(df_ru, df_zh, b6_strat),
    '6.2–6.3 Персон.+Интертекст': ipm_vectors(df_ru, df_zh, b6_all),
}

summary_rows = []
for block, (rv, cv) in block_data.items():
    corr, p = spearmanr(rv, cv)
    summary_rows.append({'Блок': block, 'r': round(corr, 3),
                         'p-value': round(p, 4),
                         'Значимость': '✓' if p < 0.05 else '✗'})
spearman_df = pd.DataFrame(summary_rows)
display(spearman_df.style.applymap(
    lambda v: 'color:green; font-weight:bold' if v == '✓' else
              ('color:gray' if v == '✗' else ''),
    subset=['Значимость']))

# Общий scatter по всем признакам
r_all_ipm, c_all_ipm = ipm_vectors(df_ru, df_zh, all_features)
corr_all, p_all = spearmanr(r_all_ipm, c_all_ipm)
print(f'\nОбщая корреляция Спирмена (все признаки, IPM): r = {corr_all:.3f},  p = {p_all:.4f}')

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(r_all_ipm, c_all_ipm, alpha=0.55, s=60,
           color='slategray', edgecolors='white', linewidths=0.5)
if len(set(r_all_ipm)) > 1:
    m, b_c = np.polyfit(r_all_ipm, c_all_ipm, 1)
    xl = np.linspace(0, r_all_ipm.max()*1.05, 200)
    ax.plot(xl, m*xl + b_c, color='red', lw=1.5, label='Линия тренда')
ax.set_xlabel('Русский сегмент (IPM)', fontsize=11)
ax.set_ylabel('Китайский сегмент (IPM)', fontsize=11)
ax.set_title('Сравнение всех признаков: RU vs ZH (IPM)', fontsize=13)
ax.text(0.03, 0.94, f'Спирмен r = {corr_all:.3f}\np = {p_all:.4f}',
        transform=ax.transAxes,
        bbox=dict(facecolor='white', edgecolor='lightgray', alpha=0.8), fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Поправка на множественное сравнение: Беньямини–Хохберг (FDR)
Все p-значения из тестов Манна–Уитни по **всем блокам** корректируются совместно.  
Порог FDR = 5% (q = 0.05). **p_adj** — скорректированное p-значение (BH).

In [ ]:
if not ALL_MW_RESULTS:
    print('⚠ Список ALL_MW_RESULTS пуст — запустите ячейки с тестами выше.')
else:
    bh_df = pd.DataFrame(ALL_MW_RESULTS).copy()
    raw_pvals = bh_df['p-value (raw)'].values
    reject, pvals_adj, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')
    bh_df['p_adj (BH)']   = np.round(pvals_adj, 6)
    bh_df['Значимо (BH)'] = reject
    bh_df['Значимость']   = bh_df['p_adj (BH)'].apply(
        lambda v: '✓ p_adj < 0.05' if v < 0.05 else '✗ p_adj ≥ 0.05')

    n_raw_sig = int((bh_df['p-value (raw)'] < 0.05).sum())
    n_bh_sig  = int(bh_df['Значимо (BH)'].sum())
    print(f'Всего тестов: {len(bh_df)}')
    print(f'Значимых до поправки  (p < 0.05):      {n_raw_sig}')
    print(f'Значимых после поправки BH (q < 0.05): {n_bh_sig}')
    print()

    display_df = bh_df[['Блок','Признак','U','p-value (raw)','p_adj (BH)','Значимость']] \
                 .sort_values('p_adj (BH)')
    display(display_df.style
        .applymap(
            lambda v: 'font-weight:bold; color:darkgreen' if '✓' in str(v) else
                      ('color:gray' if '✗' in str(v) else ''),
            subset=['Значимость'])
        .format({'p-value (raw)':'{:.5f}','p_adj (BH)':'{:.5f}','U':'{:.1f}'})
        .background_gradient(subset=['p_adj (BH)'], cmap='RdYlGn_r', vmin=0, vmax=0.1))

In [ ]:
if ALL_MW_RESULTS:
    sig_bh = bh_df[bh_df['Значимо (BH)']].sort_values('p_adj (BH)')
    if sig_bh.empty:
        print('После поправки BH значимых признаков не осталось.')
    else:
        fig, ax = plt.subplots(figsize=(10, max(4, len(sig_bh) * 0.45)))
        y = np.arange(len(sig_bh))
        clrs = ['#d62728' if p < 0.01 else '#ff7f0e' for p in sig_bh['p_adj (BH)']]
        ax.barh(y, -np.log10(sig_bh['p_adj (BH)']), color=clrs, edgecolor='white')
        ax.axvline(-np.log10(0.05), color='black', linestyle='--', lw=1.2,
                   label='FDR = 0.05 (−log₁₀ = 1.30)')
        ax.set_yticks(y)
        ax.set_yticklabels(sig_bh['Признак'], fontsize=9)
        ax.set_xlabel('−log₁₀(p_adj)', fontsize=11)
        ax.set_title(f'Значимые признаки после поправки BH (n={len(sig_bh)})', fontsize=13)
        ax.legend(fontsize=9)
        plt.tight_layout(); plt.show()

        print('\n=== Значимые признаки (p_adj < 0.05) ===')
        display(sig_bh[['Блок','Признак','p-value (raw)','p_adj (BH)']].reset_index(drop=True))

---
## Scatter-plots корреляции по блокам (IPM)

In [ ]:
for block_name, cols in [
    ('Блок 1.1–1.2: Графика и пунктуация', b1_all),
    ('Блок 1.3: Эмотиконы', b1_emoji),
    ('Блок 1.4–1.5: Смешение кодов', b1_codes),
    ('Блок 2: Фонетический уровень', b2),
    ('Блок 3: Морфология', b3_morph),
    ('Блок 4: Лексико-семантический', b4_style + b4_borr + b4_eval + b4_add),
    ('Блок 5: Синтаксический', b5_struct + b5_goal + b5_expr),
    ('Блок 6: Прагматический', b6_strat + b6_all),
]:
    plot_block_scatter(*ipm_vectors(df_ru, df_zh, cols),
                       [c.replace('\n',' ').strip() for c in cols],
                       block_name)